# RISSK — scheduled pipeline driver (JupyterHub Notebook Jobs)

Headless replacement for the legacy `rissk_readme.ipynb`. All logic lives in
`rissk_kedro.driver.run`; this notebook only chooses **which run-configuration
file** to execute. Given that file, the driver syncs the Survey Solutions export
zips down from S3, runs the selected Kedro pipelines (in-process, no command
line) questionnaire by questionnaire, syncs the generated stages back to the
survey's S3 folder, and optionally cleans the local data.

**Prerequisites**

- Environment installed from the repo root: `conda env create -f environment.yml`
  (or `uv sync`). Both workspace packages are installed editable.
- Kernel registered so Notebook Jobs can run on it:
  `python -m ipykernel install --user --name rissk_kedro`
- [AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html)
  installed and set up with credentials for the bucket. If you do not need the S3
  sync, set `SYNC_DOWN` / `SYNC_UP` to `false` in the run-config file.

**Configuring runs — no notebook edits needed**

Everything is set in a per-survey YAML under
[notebooks/configs/](notebooks/configs/) (the equivalent of the legacy `env.yaml`): the S3
survey folder, the questionnaires/versions, **which pipelines to run**, and the
sync/clean flags. See [notebooks/configs/example.yaml](notebooks/configs/example.yaml) for the
template.

The cell below is tagged `parameters`, so to schedule a different survey you only
override `CONFIG_FILE` in the Notebook Jobs *Parameters* form (e.g.
`CONFIG_FILE = "notebooks/configs/fbf.yaml"`) — one job per survey, same
notebook, no code changes.

**Running several surveys.** Prefer **one Notebook Job per config file** —
independent schedules, retries and logs, and they can run in parallel. To process
several in a *single* run instead, set `CONFIG_FILE` to a list; the driver runs
them in sequence:

```python
CONFIG_FILE = ["notebooks/configs/grenada.yaml", "notebooks/configs/hies2024.yaml"]
```

In [ ]:
# Which run-configuration file to execute.
# Path is relative to the repo root, or absolute. See notebooks/configs/.
CONFIG_FILE = "notebooks/configs/hies2024.yaml"

In [ ]:
from rissk_kedro.driver import run

run(CONFIG_FILE)

### Where results land

Each questionnaire is uploaded to its own subfolder of the survey's S3 prefix:

```
s3://<bucket>/<SURVEY>/latest/<questionnaire>/
    20_INTERIM/   ...
    30_PROCESSED/ ...
    40_SCORED/    item_scores.parquet, responsible_scores.csv, unit_rissk_scores.csv
```

The per-questionnaire subfolder is required because the Kedro output filenames are
questionnaire-agnostic (`unit_rissk_scores.csv`, …): uploading two questionnaires of the
same survey into one flat folder would overwrite each other. The final scores file is
`40_SCORED/unit_rissk_scores.csv` (the legacy pipeline wrote `results/unit_risk_score.csv`)
— point downstream consumers there.